# Facility / Component Label Viewer

Pick a **component** and a **facility** below, and this notebook prints every
labelled image for that combination — the *full original image* (not a crop),
with each annotated region outlined and its current label drawn directly on
top of the region, plus a row of buttons under each image to relabel any
region.

Like `relabel.ipynb`, edits are never written to the original files under
`Labels/`. The first time an image's label file is touched, a working copy is
created under that facility's `relabel/` folder and every edit — from this
notebook or from `relabel.ipynb` — is read from and written to that same
copy. This notebook always reads from the `relabel/` copy (creating it on
first use), so you're always viewing/editing the current working state, not
the pristine original.

Run this notebook from the same folder as `prep.py` (it imports `prep` for
the facility/class config so the two never drift apart).

In [1]:
# --- Configuration: the two variables you change ---

COMPONENT = "clarifier"   # "clarifier" or "aerobic_zone"
FACILITY = "CapeFlats"    # Atlantis | CapeFlats | Waterval | Fisantekraal | NoordelikeWerke

# --- Display / paging options (usually fine as-is) ---
DISPLAY_MAX_DIM = 1400    # longest side, in px, images are downscaled to for display
GRID_COLS = 1               # images per row (single column, larger images)
BATCH_SIZE = 8              # images per page
FONT_SIZE = 60               # label font size, drawn at full resolution before downscaling

# How to detect "this is the same image as one I've already queued":
#   "content"  -- hash the file bytes (catches an identical file copied into
#                 more than one unit's folder, even under a different name)
#   "filename" -- just compare filenames (catches the same base filename
#                 reused across units even if the bytes differ slightly,
#                 e.g. a re-compressed or re-exported copy)
DEDUP_BY = "content"


In [2]:
import hashlib
import io
import json
import shutil
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont
import ipywidgets as widgets
from IPython.display import display

import prep  # reuses FACILITIES / classes / attr keys / helpers from prep.py

# --- Validate config ---
if COMPONENT not in ("aerobic_zone", "clarifier"):
    raise ValueError(f"COMPONENT must be 'aerobic_zone' or 'clarifier', got {COMPONENT!r}")
if FACILITY not in prep.FACILITIES:
    raise ValueError(f"FACILITY must be one of {sorted(prep.FACILITIES)}, got {FACILITY!r}")

ATTR_KEY = prep.AEROBIC_ATTR_KEY if COMPONENT == "aerobic_zone" else prep.CLARIFIER_ATTR_KEY
JSON_KEY = "aerobic_json" if COMPONENT == "aerobic_zone" else "clarifier_json"

# Every class prep.py knows about for this component (including excluded ones
# like 'Stagnant'/'Empty' aerobic) is selectable here, since this notebook is
# for correcting ground truth, not just reviewing what training will use.
RELABEL_CLASSES = sorted(
    prep.AEROBIC_CLASSES if COMPONENT == "aerobic_zone" else prep.CLARIFIER_CLASSES
)

facility_root = prep.RAW_DATA_ROOT / FACILITY
facility_cfg = prep.FACILITIES[FACILITY]

print(f"Component: {COMPONENT} (attr key: '{ATTR_KEY}')")
print(f"Facility:  {FACILITY}  ({facility_root})")
print(f"Relabel classes: {RELABEL_CLASSES}")

# --- Font for on-image labels (drawn at full resolution, before downscaling) ---
_FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    "DejaVuSans-Bold.ttf",
]
FONT = None
for _fp in _FONT_CANDIDATES:
    try:
        FONT = ImageFont.truetype(_fp, FONT_SIZE)
        break
    except OSError:
        continue
if FONT is None:
    FONT = ImageFont.load_default()
    print("[warn] no TrueType font found -- labels will use a small bitmap font. "
          "Install 'fonts-dejavu-core' (or similar) for larger, clearer labels.")


Component: clarifier (attr key: 'clarifier')
Facility:  CapeFlats  (../wastewater/CapeFlats)
Relabel classes: ['Dysfunctional', 'Empty', 'Functional', 'Scum', 'Stagnant']


In [3]:
# --- relabel/ working-copy helpers ---
# Mirrors relabel.ipynb: the first time a given json file is touched, a copy
# is made under facility_root/relabel/{name}.json and every read/write after
# that goes through that copy, leaving Labels/ untouched.

def get_relabel_dir() -> Path:
    d = facility_root / prep.RELABEL_SUBDIR
    d.mkdir(parents=True, exist_ok=True)
    return d


def get_working_copy_path(json_filename: str) -> Path:
    working = get_relabel_dir() / json_filename
    if not working.exists():
        original = facility_root / prep.LABELS_SUBDIR / json_filename
        if not original.exists():
            raise FileNotFoundError(f"Neither {working} nor {original} exists.")
        shutil.copy2(original, working)
        print(f"[info] created relabel working copy: {working}")
    return working


def load_entry(working_copy_path: Path, img_key: str) -> dict:
    """Re-reads the working copy from disk and returns the entry for img_key,
    so every draw/render always reflects the latest saved edit."""
    img_metadata = prep.load_via2_json(working_copy_path)
    return img_metadata[img_key]


def apply_relabel(working_copy_path: Path, img_key: str, region_index: int, new_label: str):
    with open(working_copy_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    img_metadata = data.get("_via_img_metadata", data)
    region = img_metadata[img_key]["regions"][region_index]
    region["region_attributes"][ATTR_KEY] = new_label
    with open(working_copy_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


In [4]:
# --- Collect every image for this component + facility, across all units ---
# (CapeFlats/Waterval have several units, each with their own json + image dir;
# some facilities have the same underlying photo copied into more than one
# unit's folder, so we dedupe -- see DEDUP_BY above for how "same image" is
# decided)

def file_content_key(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


def region_labels(entry):
    return sorted(
        r.get("region_attributes", {}).get(ATTR_KEY)
        for r in entry.get("regions", [])
        if r.get("region_attributes", {}).get(ATTR_KEY)
    )


def collect_items():
    items = []
    seen = {}      # dedup_key -> {origin, labels} for the item that first claimed it
    skipped = []    # (dup_unit, dup_filename, origin, conflict_labels_or_None) for the diagnostic report below

    for unit in facility_cfg["units"]:
        json_name = unit.get(JSON_KEY)
        if not json_name:
            continue  # this unit has no json for the chosen component (e.g. aerobic at Fisantekraal)

        unit_img_dir = unit.get("images_dir")
        unit_label = unit_img_dir if unit_img_dir else FACILITY
        images_dir = facility_root / unit_img_dir if unit_img_dir else facility_root

        working_copy_path = get_working_copy_path(json_name)
        img_metadata = prep.load_via2_json(working_copy_path)

        for img_key, entry in img_metadata.items():
            filename = entry.get("filename")
            if not filename:
                continue

            image_path = prep.find_image_case_insensitive(images_dir, filename)
            if image_path is None:
                continue  # missing on disk -- prep.py's own diagnostic report covers this

            has_component_region = any(
                r.get("region_attributes", {}).get(ATTR_KEY)
                for r in entry.get("regions", [])
            )
            if not has_component_region:
                continue

            dedup_key = file_content_key(image_path) if DEDUP_BY == "content" else filename.lower()
            labels = region_labels(entry)

            if dedup_key in seen:
                original = seen[dedup_key]
                conflict = labels if labels != original["labels"] else None
                skipped.append((unit_label, filename, original["origin"], original["labels"], conflict))
                continue
            seen[dedup_key] = {"origin": f"{unit_label}/{filename}", "labels": labels}

            items.append({
                "unit_label": unit_label,
                "images_dir": images_dir,
                "working_copy_path": working_copy_path,
                "img_key": img_key,
                "image_path": image_path,
                "filename": filename,
            })

    if skipped:
        print(f"Skipped {len(skipped)} duplicate(s) (DEDUP_BY='{DEDUP_BY}'):")
        for dup_unit, dup_filename, origin, kept_labels, conflict in skipped:
            print(f"  - {dup_unit}/{dup_filename}  ==  {origin}")
            if conflict is not None:
                print(f"      \u26a0\ufe0f CONFLICT: kept labels {kept_labels} but dropped entry had {conflict} -- check this one by hand")

    return items


items = collect_items()
n_batches = max(1, -(-len(items) // BATCH_SIZE))  # ceil div
print(f"Found {len(items)} unique images with '{COMPONENT}' labels for facility '{FACILITY}'.")


Found 368 unique images with 'clarifier' labels for facility 'CapeFlats'.


In [5]:
# --- Draw the full image with each region outlined + its label on top ---

TEXT_COLOR = (255, 255, 0)
TEXT_BG = (0, 0, 0)


def draw_annotated_image(item) -> bytes:
    entry = load_entry(item["working_copy_path"], item["img_key"])
    img = Image.open(item["image_path"]).convert("RGB")
    draw = ImageDraw.Draw(img)

    for i, region in enumerate(entry.get("regions", [])):
        shape_attrs = region.get("shape_attributes", {})
        label = region.get("region_attributes", {}).get(ATTR_KEY)
        if not label:
            continue

        shape = shape_attrs.get("name")
        if shape == "circle":
            cx, cy, r = shape_attrs["cx"], shape_attrs["cy"], shape_attrs["r"]
            text_xy = (cx - r, cy - r - FONT_SIZE - 14)
        elif shape == "ellipse":
            cx, cy, rx, ry = shape_attrs["cx"], shape_attrs["cy"], shape_attrs["rx"], shape_attrs["ry"]
            text_xy = (cx - rx, cy - ry - FONT_SIZE - 14)
        elif shape == "rect":
            x, y, w, h = shape_attrs["x"], shape_attrs["y"], shape_attrs["width"], shape_attrs["height"]
            text_xy = (x, y - FONT_SIZE - 14)
        else:
            continue

        tx, ty = max(0, text_xy[0]), max(0, text_xy[1])
        text = f"[{i}] {label}"
        bbox = draw.textbbox((tx, ty), text, font=FONT)
        draw.rectangle(bbox, fill=TEXT_BG)
        draw.text((tx, ty), text, fill=TEXT_COLOR, font=FONT)

    scale = min(1.0, DISPLAY_MAX_DIM / max(img.width, img.height))
    if scale < 1.0:
        img = img.resize((max(1, int(img.width * scale)), max(1, int(img.height * scale))), Image.LANCZOS)

    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def make_item_widget(item):
    img_widget = widgets.Image(value=draw_annotated_image(item), format="png")
    filename_label = widgets.HTML(f"<b>{item['unit_label']} / {item['filename']}</b>")

    entry = load_entry(item["working_copy_path"], item["img_key"])
    region_rows = []

    for i, region in enumerate(entry.get("regions", [])):
        if not region.get("region_attributes", {}).get(ATTR_KEY):
            continue  # region belongs to a different component's annotation pass

        btns = {}

        def refresh_row_styles(region_index=i, btns=btns):
            fresh = load_entry(item["working_copy_path"], item["img_key"])
            current = fresh["regions"][region_index]["region_attributes"].get(ATTR_KEY)
            for cls_name, b in btns.items():
                b.style.button_color = "#4caf50" if cls_name == current else None
                b.description = f"\u2713 {cls_name}" if cls_name == current else cls_name

        def make_handler(region_index):
            def _handler(_):
                clicked_label = _handler.cls_name
                apply_relabel(item["working_copy_path"], item["img_key"], region_index, clicked_label)
                img_widget.value = draw_annotated_image(item)
                refresh_row_styles()
            return _handler

        row_children = [widgets.Label(value=f"Region {i}:", layout=widgets.Layout(width="70px"))]
        for cls_name in RELABEL_CLASSES:
            b = widgets.Button(description=cls_name, layout=widgets.Layout(width="100px"))
            handler = make_handler(i)
            handler.cls_name = cls_name
            b.on_click(handler)
            btns[cls_name] = b
            row_children.append(b)

        refresh_row_styles()
        region_rows.append(widgets.HBox(row_children))

    return widgets.VBox(
        [filename_label, img_widget] + region_rows,
        layout=widgets.Layout(border="1px solid #ccc", padding="6px", margin="4px",
                               width=f"{DISPLAY_MAX_DIM + 20}px"),
    )


In [ ]:
# --- Paging controls + display ---

output_area = widgets.Output()
batch_label = widgets.Label()
state = {"batch_idx": 0}


def render_batch(idx):
    idx = max(0, min(idx, n_batches - 1))
    state["batch_idx"] = idx

    start = idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(items))
    batch_items = items[start:end]

    batch_label.value = f"Batch {idx + 1}/{n_batches}  (items {start + 1}-{end} of {len(items)})"

    item_widgets = [make_item_widget(it) for it in batch_items]
    grid = widgets.GridBox(
        item_widgets,
        layout=widgets.Layout(grid_template_columns=f"repeat({GRID_COLS}, auto)"),
    )

    with output_area:
        output_area.clear_output(wait=True)
        display(grid)


prev_btn = widgets.Button(description="\u2b05 Previous")
next_btn = widgets.Button(description="Next \u27a1")
prev_btn.on_click(lambda _: render_batch(state["batch_idx"] - 1))
next_btn.on_click(lambda _: render_batch(state["batch_idx"] + 1))

controls = widgets.HBox([prev_btn, batch_label, next_btn])
display(widgets.VBox([controls, output_area]))

if items:
    render_batch(0)
else:
    print("No images found for this component/facility combination -- double-check COMPONENT and FACILITY above.")
